# Baseline Evaluation
This notebook evaluates the source model on APTOS dataset without adaptation.

Measures the domain gap between IDRiD (source) and APTOS (target).

In [1]:
import sys, os; sys.path.insert(0, os.getcwd())
import os, sys
_d = os.getcwd()
PROJECT_ROOT = None
while _d != os.path.dirname(_d):
    if os.path.exists(os.path.join(_d, 'setup.py')):
        PROJECT_ROOT = _d
        break
    _d = os.path.dirname(_d)
if PROJECT_ROOT is None:
    for _p in ['/kaggle/working/medical_CTTA', '/content/medical_CTTA']:
        if os.path.isdir(_p):
            PROJECT_ROOT = _p
            break
if PROJECT_ROOT and PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.env import init
init()

from src.config import load_config

In [2]:
# Load config
config = load_config('configs/default.yaml')
print(f'Loading trained source model from: {config.output_dir}')

# HuggingFace token (needed for RETFound gated weights)
if not os.environ.get('HF_TOKEN'):
    import getpass
    token = getpass.getpass('Enter your HuggingFace token (hf_...): ')
    os.environ['HF_TOKEN'] = token
    print('✓ HF_TOKEN set')

Loading trained source model from: ./outputs/
✓ HF_TOKEN set


In [3]:
# Evaluate source model on both IDRiD and APTOS
from src.evaluation.runner import CTTARunner

runner = CTTARunner(config)
model = runner._load_model()

# Evaluate on source
source_results = runner._evaluate_model(model, config.source_dataset, train=False)
print(f'Source (IDRiD) QWK: {source_results["qwk"]:.4f}')

# Evaluate on target (no adaptation - this is the domain gap)
target_results = runner._evaluate_model(model, config.target_dataset, train=False)
print(f'Target (APTOS) QWK (no adaptation): {target_results["qwk"]:.4f}')

# Domain gap
gap = source_results['qwk'] - target_results['qwk']
print(f'\nDomain gap: {gap:.4f}')

2026-07-13 17:46:27,835 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/YukunZhou/RETFound_mae_natureCFP/resolve/main/RETFound_mae_natureCFP.pth "HTTP/1.1 302 Found"
2026-07-13 17:46:44,570 - src.evaluation.runner - INFO - Loading source checkpoint
2026-07-13 17:46:48,078 - src.evaluation.runner - INFO - Loaded source classifier checkpoint
Source (IDRiD) QWK: 0.6492
Target (APTOS) QWK (no adaptation): 0.6492

Domain gap: 0.0000
